本文档用于协助开发Behavior Prompt在Tbot上，主要是在开发过程中，验证每个模块的功能性，支持开发进行。

例如，加载LerobotDataset的扩展类，检测是否可用于数据加载，以及实现了数据处理管道

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds = LeRobotDataset('/vla/workspace/data/hanging_mug/aloha-agilex_clean_50')
ds

In [ ]:
from lerobot.datasets.behavior_prompt_dataset import BehaviorPromptLeRobotDataset,BehaviorPromptConfig
config = BehaviorPromptConfig()
config.prompt_action_chunk_size = 50
config.max_prompt_chunks = None
config.same_episode_policy = "avoid"
config.seed = 0

bp_ds = BehaviorPromptLeRobotDataset(ds, ds, config)
bp_ds

/vla/.conda/miniconda3/envs/mytbot/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
bp_ds.transform

IdentityTransformFn()

In [5]:
from types import SimpleNamespace

from lerobot.policies.BP_TBot.configuration_bp_tbot import BPTBotConfig, BPTBotDatasetConfig

# 从当前已经加载好的 ds 里取 dataset 信息，避免手写 repo/root 出错
bp_repo_id = getattr(ds, "repo_id", None)
bp_root = getattr(ds, "root", None)
bp_revision = getattr(ds, "revision", None)
bp_video_backend = getattr(ds, "video_backend", None)

# 如果你的 ds 没有 repo_id 属性，可以手动指定：
# bp_repo_id = "your/repo_id"

if bp_repo_id is None:
    raise ValueError("当前 ds 没有 repo_id，请手动设置 bp_repo_id")

bp_dataset_cfg = BPTBotDatasetConfig(
    repo_id=bp_repo_id,
    root=bp_root,
    revision=bp_revision,
    video_backend=bp_video_backend,
    episodes=getattr(ds, "episodes", None),
    action_mode="delta",
    bp_num_chunks=4,
    bp_same_episode_policy="avoid",
    bp_seed=0,
    height=224,
    width=224,
    max_state_dim=32,
    max_action_dim=32,
    qwen3_vl_processor_path="/vla/workspace/models/Qwen3-VL-2B-Instruct",
)

bp_policy_cfg = BPTBotConfig(
    device=None,
    chunk_size=50,
    n_action_steps=50,
    n_obs_steps=1,
    max_state_dim=32,
    max_action_dim=32,
    image_delta_indices=[-15, 0, 15],
    bp_num_chunks=4,
    bp_action_chunk_size=50,
)

cfg = SimpleNamespace(
    dataset=bp_dataset_cfg,
    policy=bp_policy_cfg,
)

cfg

namespace(dataset=BPTBotDatasetConfig(repo_id='/vla/workspace/data/hanging_mug/aloha-agilex_clean_50', repo_id_file=None, root=PosixPath('/vla/workspace/data/hanging_mug/aloha-agilex_clean_50'), episodes=None, image_transforms=ImageTransformsConfig(enable=False, preset=None, max_num_transforms=3, random_order=False, tfs={'brightness': ImageTransformConfig(weight=1.0, type='ColorJitter', kwargs={'brightness': (0.8, 1.2)}), 'contrast': ImageTransformConfig(weight=1.0, type='ColorJitter', kwargs={'contrast': (0.8, 1.2)}), 'saturation': ImageTransformConfig(weight=1.0, type='ColorJitter', kwargs={'saturation': (0.5, 1.5)}), 'hue': ImageTransformConfig(weight=1.0, type='ColorJitter', kwargs={'hue': (-0.05, 0.05)}), 'sharpness': ImageTransformConfig(weight=1.0, type='SharpnessJitter', kwargs={'sharpness': (0.5, 1.5)}), 'affine': ImageTransformConfig(weight=1.0, type='RandomAffine', kwargs={'degrees': (-5.0, 5.0), 'translate': (0.05, 0.05)})}), revision='v3.0', use_imagenet_stats=True, use_ex

In [6]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.datasets.factory import resolve_delta_timestamps, _configure_vision_only_dataset
from lerobot.datasets.behavior_prompt_dataset import BehaviorPromptConfig, BehaviorPromptLeRobotDataset

repo_id = cfg.dataset.repo_id

ds_meta = LeRobotDatasetMetadata(
    repo_id,
    root=cfg.dataset.root,
    revision=cfg.dataset.revision,
)

delta_timestamps = resolve_delta_timestamps(cfg.policy, ds_meta)

current_ds = LeRobotDataset(
    repo_id,
    root=cfg.dataset.root,
    episodes=cfg.dataset.episodes,
    delta_timestamps=delta_timestamps,
    image_transforms=None,
    revision=cfg.dataset.revision,
    video_backend=cfg.dataset.video_backend,
)

frame_ds = LeRobotDataset(
    repo_id,
    root=cfg.dataset.root,
    episodes=cfg.dataset.episodes,
    image_transforms=None,
    revision=cfg.dataset.revision,
    video_backend=cfg.dataset.video_backend,
)

_configure_vision_only_dataset(current_ds, cfg.policy)

prompt_cfg = BehaviorPromptConfig(
    prompt_action_chunk_size=cfg.policy.bp_action_chunk_size,
    same_episode_policy=cfg.dataset.bp_same_episode_policy,
    seed=cfg.dataset.bp_seed,
    num_chunks=cfg.dataset.bp_num_chunks,
    height=cfg.dataset.height,
    width=cfg.dataset.width,
    max_state_dim=cfg.dataset.max_state_dim,
    max_action_dim=cfg.dataset.max_action_dim,
    qwen3_vl_processor_path=cfg.dataset.qwen3_vl_processor_path,
    action_mode=cfg.dataset.action_mode,
)

bp_ds_raw = BehaviorPromptLeRobotDataset(current_ds, frame_ds, prompt_cfg)
bp_ds_tfns = BehaviorPromptLeRobotDataset.with_default_transforms(current_ds, frame_ds, prompt_cfg)

raw_sample = bp_ds_raw[1]
tf_sample = bp_ds_tfns[1]

raw_sample.keys(), tf_sample.keys()

Hydrating transform InjectMissingStateActionTransformFn (robot_type=aloha, resolved=aloha, action_seq_len=1, state_seq_len=1, placeholder_dim=14)
Hydrating transform NormalizeTransformFn with dataset.meta.stats (robot_type=aloha, resolved=aloha) and selected_keys (selected_keys=['observation.state', 'action'])
Hydrating transform ComposeFieldsTransform with mapping (robot_type=aloha, resolved=aloha)
Hydrating transform DeltaActionTransformFn with mapping and mask (robot_type=aloha, resolved=aloha)
Hydrating transform RemapImageKeyTransformFn with mapping (robot_type=aloha, resolved=aloha)


(dict_keys(['observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist', 'observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'action_is_pad', 'observation.images.cam_high_is_pad', 'observation.images.cam_left_wrist_is_pad', 'observation.images.cam_right_wrist_is_pad', 'task', 'robot_type', 'behavior_prompt']),
 dict_keys(['observation.state', 'action', 'sample.action_loss_mask', 'observation.images.image0', 'observation.images.image1', 'observation.images.image2', 'observation.images.image0_mask', 'observation.images.image1_mask', 'observation.images.image2_mask', 'observation.pixel_values', 'observation.image_grid_thw', 'observation.input_ids', 'observation.attention_mask', 'behavior_prompt']))

In [7]:
print(bp_ds_tfns[1]['observation.pixel_values'].shape)
print(bp_ds_tfns[1]['observation.input_ids'].shape)

torch.Size([768, 1536])
torch.Size([198])


In [8]:
bp_ds_tfns.transform.transforms

[BPPadOrSampleChunksFn(num_chunks=4),
 BPResizeImagesWithPadFn(height=224, width=224, mode='bilinear'),
 BPRemapImageKeyTransformFn(mapping={'observation.images.cam_high': 'observation.images.image0', 'observation.images.cam_left_wrist': 'observation.images.image1', 'observation.images.cam_right_wrist': 'observation.images.image2'}),
 BPComposeFieldsTransform(mapping={'observation.state': ['observation.state'], 'action': ['action']}),
 BPDeltaActionTransformFn(mask=tensor([ True,  True,  True,  True,  True,  True, False,  True,  True,  True,
          True,  True,  True, False])),
 BPNormalizeTransformFn(selected_keys=['observation.state', 'action'], mode='mean_std', norm_stats={'observation.images.cam_left_wrist': {'min': array([[[0.        ]],
 
        [[0.00784314]],
 
        [[0.00392157]]]), 'max': array([[[1.]],
 
        [[1.]],
 
        [[1.]]]), 'mean': array([[[0.79589583]],
 
        [[0.76986596]],
 
        [[0.76368147]]]), 'std': array([[[0.29336312]],
 
        [[0.2

# Dataloader

In [9]:
# 基于 bp_ds_tfns 构造一个接近训练 DataLoader 输出的 batch
# 这里先用 PyTorch 默认 collate，验证 nested behavior_prompt 是否能正常 batch 化。

from torch.utils.data import DataLoader
from torch.utils.data._utils.collate import default_collate
import torch

bp_batch_size = 2
bp_batch_indices = [1, 2]

bp_samples = [bp_ds_tfns[i] for i in bp_batch_indices]
bp_batch = default_collate(bp_samples)


def print_tree_shapes(x, prefix=""):
    """递归打印 dict/list/tensor 的 shape，方便检查 batch schema。"""
    if isinstance(x, dict):
        for key, value in x.items():
            next_prefix = f"{prefix}.{key}" if prefix else str(key)
            print_tree_shapes(value, next_prefix)
    elif isinstance(x, torch.Tensor):
        print(f"{prefix:64s} shape={tuple(x.shape)} dtype={x.dtype} device={x.device}")
    else:
        print(f"{prefix:64s} type={type(x).__name__} value={x}")

print("========== BP batch keys ==========")
print(bp_batch.keys())
print("\n========== Current branch ==========")
for key in [
    "observation.state",
    "action",
    "sample.action_loss_mask",
    "observation.pixel_values",
    "observation.image_grid_thw",
    "observation.input_ids",
    "observation.attention_mask",
]:
    print_tree_shapes(bp_batch[key], key)

print("\n========== Behavior prompt branch ==========")
print_tree_shapes(bp_batch["behavior_prompt"], "behavior_prompt")

# 关键 shape 断言：后续可以直接把 bp_batch 喂给 BPTBotPolicy.forward。
assert bp_batch["behavior_prompt"]["state"].ndim == 3          # (B, K, state_dim)
assert bp_batch["behavior_prompt"]["action"].ndim == 4         # (B, K, T, action_dim)
assert bp_batch["behavior_prompt"]["mask"].shape[:2] == bp_batch["behavior_prompt"]["state"].shape[:2]
assert bp_batch["observation.input_ids"].ndim == 2             # (B, current_visual_tokens)
bp_batch

========== BP batch keys ==========
dict_keys(['observation.state', 'action', 'sample.action_loss_mask', 'observation.images.image0', 'observation.images.image1', 'observation.images.image2', 'observation.images.image0_mask', 'observation.images.image1_mask', 'observation.images.image2_mask', 'observation.pixel_values', 'observation.image_grid_thw', 'observation.input_ids', 'observation.attention_mask', 'behavior_prompt'])

========== Current branch ==========
observation.state                                                shape=(2, 32) dtype=torch.float32 device=cpu
action                                                           shape=(2, 50, 32) dtype=torch.float32 device=cpu
sample.action_loss_mask                                          shape=(2, 1) dtype=torch.float32 device=cpu
observation.pixel_values                                         shape=(2, 768, 1536) dtype=torch.float32 device=cpu
observation.image_grid_thw                                       shape=(2, 3, 3) dtyp

{'observation.state': tensor([[ 0.7823, -0.9669, -0.9586,  0.9717,  0.0818, -0.0081,  0.5293, -0.7658,
          -0.9896, -0.9367, -0.1082, -0.0020, -0.1947,  0.6564,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.7823, -0.9669, -0.9586,  0.9717,  0.0818, -0.0081,  0.5293, -0.7658,
          -0.9896, -0.9367, -0.1082, -0.0020, -0.1947,  0.6564,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]]),
 'action': tensor([[[ 0.7820, -0.9669, -0.9586,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.7745, -0.9568, -0.9471,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.7590, -0.9359, -0.9234,  ...,  0.0000,  0.0000,  0.0000],
          ...,
          [-0.3463,  1.0870,  0.8131,  ...,  0.0000,  0.0000,  0.0000],
      

# BP_TBot policy init/save/reload forward check

这一节从已经构造好的 `bp_batch` 出发，做三件事：

1. 用 `BPTBotPolicy.from_pretrained('/vla/workspace/models/tbot_base', config=bp_policy_cfg)` 加载原始 TBot-SA1 权重，并随机初始化 BP 新增层。
2. 跑一次完整 `policy.forward(bp_batch)`，确认当前 BP forward 可以执行。
3. 保存为 `/vla/workspace/models/bp_tbot_init`，再从这个目录重新加载并再次 forward，确认后续可以直接加载 BP 初始权重。

In [10]:
from pathlib import Path
import copy
import torch

from lerobot.policies.BP_TBot.modeling_bp_tbot import BPTBotPolicy

# 需要先执行前面的 cfg / bp_batch 构造单元。
assert "cfg" in globals(), "请先执行上面的 cfg 构造单元"
assert "bp_batch" in globals(), "请先执行上面的 Dataloader/bp_batch 构造单元"

bp_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bp_tbot_base_path = Path("/vla/workspace/models/tbot_ckpts/MytbotBase/30w/pretrained_model")
bp_tbot_init_path = Path("/vla/workspace/models/bp_mytbot_init_test")

# notebook 前面的 cfg.policy 可能是在旧代码状态下创建的，这里显式对齐 tbot_base checkpoint。
# 路径我修复了，agent你不要再改了
bp_qwen_path = Path("/vla/workspace/models/Qwen3-VL-2B-Instruct")
bp_cosmos_path = Path("/vla/workspace/models/Cosmos-Tokenizer-CI8x8")
bp_da3_model_path = Path("/vla/workspace/models/DA3-LARGE-1.1")
bp_da3_code_root = Path("/vla/workspace/my_tbot/third_party/Depth-Anything-3")

for name, path in [
    ("tbot_base", bp_tbot_base_path),
    ("Qwen3-VL", bp_qwen_path),
    ("Cosmos tokenizer", bp_cosmos_path),
    ("DA3 model", bp_da3_model_path),
    ("DA3 code root", bp_da3_code_root),
]:
    if not path.exists():
        raise FileNotFoundError(f"{name} path does not exist: {path}")

bp_policy_cfg = copy.deepcopy(cfg.policy)
bp_policy_cfg.device = str(bp_device)
bp_policy_cfg.pretrained_path = None
bp_policy_cfg.qwen3_vl_variant = "qwen3_vl_28l"
bp_policy_cfg.action_expert_variant = "qwen3_28l"
bp_policy_cfg.qwen3_vl_pretrained_path = str(bp_qwen_path)
bp_policy_cfg.cosmos_tokenizer_path_or_name = str(bp_cosmos_path)
bp_policy_cfg.enable_3d_queries = True
bp_policy_cfg.num_3d_query_tokens = 432
bp_policy_cfg.lambda_3d = 0.01
bp_policy_cfg.da3_model_path_or_name = str(bp_da3_model_path)
bp_policy_cfg.da3_code_root = str(bp_da3_code_root)
bp_policy_cfg.log_da3_teacher_timing = True
bp_policy_cfg.bp_num_chunks = 7
bp_policy_cfg.bp_action_chunk_size = 50
bp_policy_cfg.bp_use_type_embedding = True
bp_policy_cfg.bp_use_chunk_embedding = True
bp_policy_cfg.bp_use_action_step_embedding = True
bp_policy_cfg.validate_features()

# forward 用 batch 放到同一 device；只移动 tensor，保留嵌套 dict 结构。
def move_to_device(x, device):
    if isinstance(x, torch.Tensor):
        return x.to(device)
    if isinstance(x, dict):
        return {k: move_to_device(v, device) for k, v in x.items()}
    if isinstance(x, list):
        return [move_to_device(v, device) for v in x]
    if isinstance(x, tuple):
        return tuple(move_to_device(v, device) for v in x)
    return x

bp_batch_for_forward = move_to_device(bp_batch, bp_device)

print("device:", bp_device)
print("tbot_base:", bp_tbot_base_path)
print("Qwen3-VL:", bp_policy_cfg.qwen3_vl_pretrained_path)
print("Cosmos tokenizer:", bp_policy_cfg.cosmos_tokenizer_path_or_name)
print("DA3 model:", bp_policy_cfg.da3_model_path_or_name)
print("DA3 code root:", bp_policy_cfg.da3_code_root)
print("bp init save path:", bp_tbot_init_path)
print("policy feature action shape:", bp_policy_cfg.output_features["action"].shape)
print("batch action shape:", tuple(bp_batch_for_forward["action"].shape))
print("BP action shape:", tuple(bp_batch_for_forward["behavior_prompt"]["action"].shape))

device: cuda
tbot_base: /vla/workspace/models/tbot_ckpts/MytbotBase/30w/pretrained_model
Qwen3-VL: /vla/workspace/models/Qwen3-VL-2B-Instruct
Cosmos tokenizer: /vla/workspace/models/Cosmos-Tokenizer-CI8x8
DA3 model: /vla/workspace/models/DA3-LARGE-1.1
DA3 code root: /vla/workspace/my_tbot/third_party/Depth-Anything-3
bp init save path: /vla/workspace/models/bp_mytbot_init_test
policy feature action shape: (32,)
batch action shape: (2, 50, 32)
BP action shape: (2, 4, 50, 32)


In [11]:
# 1) 从原始 TBot-SA1 base checkpoint 初始化 BP_TBot。
#    原 checkpoint 没有 bp_* 新层；这些层会作为 expected missing keys 随机初始化。
torch.manual_seed(0)
if bp_device.type == "cuda":
    torch.cuda.manual_seed_all(0)

bp_policy_from_base = BPTBotPolicy.from_pretrained(
    str(bp_tbot_base_path),
    config=bp_policy_cfg,
    strict=False,
)
bp_policy_from_base.train()

# 完整 policy.forward：会走 BP prefix、middle/suffix、loss_action、loss_gen、loss_3d。
torch.manual_seed(123)
if bp_device.type == "cuda":
    torch.cuda.manual_seed_all(123)
with torch.no_grad():
    bp_loss_from_base, bp_loss_dict_from_base = bp_policy_from_base.forward(bp_batch_for_forward)

print("initialized from tbot_base")
print("loss:", float(bp_loss_from_base.item()))
for key in ["loss", "loss_action", "loss_gen", "loss_3d"]:
    print(f"{key}: {bp_loss_dict_from_base[key]}")

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
[INFO ] using MLP layer as FFN
Loading weights from local directory
Loading weights from local directory
[INFO ] Selecting reference view using strategy: saddle_balanced
initialized from tbot_base
loss: 0.35008126497268677
loss: 0.35008126497268677
loss_action: 0.31880906224250793
loss_gen: 2.5830113887786865
loss_3d: 0.5442084074020386


In [12]:
# 2) 保存 BP_TBot 初始权重。
#    保存后 config.json 的 type 是 BP_TBot，且包含 bp_* 新层权重，后续可以直接 from_pretrained 加载。
bp_tbot_init_path.mkdir(parents=True, exist_ok=True)

# save_pretrained 会写 config.json 和 model.safetensors。
bp_policy_from_base.save_pretrained(bp_tbot_init_path)

print("saved BP_TBot init checkpoint:", bp_tbot_init_path)
print("files:", sorted(p.name for p in bp_tbot_init_path.iterdir()))

saved BP_TBot init checkpoint: /vla/workspace/models/bp_mytbot_init_test
files: ['config.json', 'model.safetensors']


In [14]:
# 3) 从刚保存的 BP_TBot 初始权重重新加载。
#    这里不再指向 tbot_base，而是加载已经包含 bp_* 参数的新 checkpoint。
bp_reload_cfg = copy.deepcopy(bp_policy_cfg)
bp_reload_cfg.pretrained_path = None
bp_reload_cfg.device = str(bp_device)

bp_policy_reloaded = BPTBotPolicy.from_pretrained(
    str('/vla/workspace/models/bp_mytbot_init_test'),
    config=bp_reload_cfg,
    strict=False,
)
bp_policy_reloaded.train()

print("reloaded BP_TBot checkpoint:", bp_tbot_init_path)
print("bp_type_embedding shape:", tuple(bp_policy_reloaded.model.bp_type_embedding.weight.shape))
print("bp_chunk_embedding shape:", tuple(bp_policy_reloaded.model.bp_chunk_embedding.weight.shape))
print("bp_action_step_embedding shape:", tuple(bp_policy_reloaded.model.bp_action_step_embedding.weight.shape))

[INFO ] using MLP layer as FFN
Loading weights from local directory
Loading weights from local directory
reloaded BP_TBot checkpoint: /vla/workspace/models/bp_mytbot_init_test
bp_type_embedding shape: (2, 2048)
bp_chunk_embedding shape: (7, 2048)
bp_action_step_embedding shape: (50, 32)


In [ ]:
# 4) 重新加载后的完整 forward 检查。
#    固定随机种子后，和保存前 forward 应该在同一数量级；若模型/随机数完全一致，loss 通常也会非常接近。
torch.manual_seed(123)
if bp_device.type == "cuda":
    torch.cuda.manual_seed_all(123)

with torch.no_grad():
    bp_loss_reloaded, bp_loss_dict_reloaded = bp_policy_reloaded.forward(bp_batch_for_forward)

print("forward after reload")
print("loss:", float(bp_loss_reloaded.item()))
for key in ["loss", "loss_action", "loss_gen", "loss_3d"]:
    print(f"{key}: {bp_loss_dict_reloaded[key]}")

print("\ncompare before_save vs after_reload")
for key in ["loss", "loss_action", "loss_gen", "loss_3d"]:
    before = float(bp_loss_dict_from_base[key])
    after = float(bp_loss_dict_reloaded[key])
    print(f"{key}: before={before:.6f}, after={after:.6f}, abs_diff={abs(before - after):.6f}")

forward after reload
loss: 0.3542476296424866
loss: 0.3542476296424866
loss_action: 0.323285847902298
loss_gen: 2.5715651512145996
loss_3d: 0.5246134400367737

compare before_save vs after_reload
loss: before=0.354248, after=0.354248, abs_diff=0.000000
loss_action: before=0.323286, after=0.323286, abs_diff=0.000000
loss_gen: before=2.571565, after=2.571565, abs_diff=0.000000
loss_3d: before=0.524613, after=0.524613, abs_diff=0.000000


In [ ]:
bp_batch_for_forward

{'observation.state': tensor([[ 1.0650, -1.0863, -0.9170,  0.4919,  0.2392, -0.2929,  0.7614, -0.6890,
          -0.6728, -0.6144,  0.4096, -0.4400,  0.5514,  0.5149,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 1.0650, -1.0863, -0.9170,  0.4919,  0.2392, -0.2929,  0.7614, -0.6890,
          -0.6728, -0.6144,  0.4096, -0.4400,  0.5514,  0.5149,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000]],
        device='cuda:0'),
 'action': tensor([[[ 1.0715, -1.0929, -0.9178,  ...,  0.0000,  0.0000,  0.0000],
          [ 1.0560, -1.0829, -0.9064,  ...,  0.0000,  0.0000,  0.0000],
          [ 1.0237, -1.0624, -0.8829,  ...,  0.0000,  0.0000,  0.0000],
          ...,
          [-1.5352,  1.0973,  0.8557,  ...,  0.0000, 

: 